In [2]:
import pandas as pd
import os
from Bio import Entrez, SeqIO
import tqdm
from collections import Counter
import numpy as np
from scipy.stats import multinomial
import numpy as np
import ast



In [3]:
# Load the data
iterator = pd.read_csv("/Users/reem/Mov/final_llrs26_per_context.tsv", sep="\t", dtype={"seqName":str, 'privateNucMutations.unlabeledSubstitutions':str } ,chunksize=1000)
df = pd.concat([chunk for chunk in tqdm.tqdm(iterator, desc='Loading data')])



Loading data: 16922it [01:11, 236.86it/s]


In [4]:
spectra = [
    {"name":"BA.1",
     "url": "https://raw.githubusercontent.com/theosanderson/molnupiravir/main/mutational_spectra/BA.1_SBS_spectrum_Ruis.csv"
    },
    {"name":"High G-to-A",
        "url": "https://raw.githubusercontent.com/theosanderson/molnupiravir/main/mutational_spectra/long_phylogenetic_branches/long_branch_spectrum_rescaled.csv"    
    },
]


In [5]:
df['privateNucMutations.unlabeledSubstitutions'].str.split(",")

0                                            [A9364G, T9901C]
1           [C455T, C1387T, G3338T, C6730T, C6936T, A13625...
2                                                         NaN
3                                                   [T16494C]
4           [C2227A, G4207T, C13673A, A21603G, C21658T, C2...
                                  ...                        
16921560                                   [C26522T, C28531T]
16921561                                             [C3241T]
16921562                            [C934T, C12784T, C26776T]
16921563                                            [C29632T]
16921564                           [G2020T, G10465A, C28419T]
Name: privateNucMutations.unlabeledSubstitutions, Length: 16921565, dtype: object

In [6]:
def fetch_reference_genome(accession='NC_045512.2'):
    Entrez.email = "theo@theo.io"  
    handle = Entrez.efetch(db="nucleotide", id=accession, rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    handle.close()
    return str(record.seq)
reference_genome = fetch_reference_genome("NC_045512.2")  # SARS-CoV-2 reference genome

In [ ]:
def get_mut_type(mut_string):
    if isinstance(mut_string, str) and len(mut_string) >= 2:
        return mut_string[0] + '>' + mut_string[-1]
    else:
        return ''
df["subs"]=df["privateNucMutations.unlabeledSubstitutions"].apply(lambda x: ','.join([get_mut_type(item) for item in x.split(',')])if isinstance(x, str) else '')
df["subs"].head()
mut_string = "C3736T,C4276T,G4422A,C6941T,C12484T,C19862T,G20925A,C28087T,C28969T"
mut_string = "C3736T,C4276T,G4422A,C6941T,C12484T,C19862T,G20925A,G23522A,C28087T,C28969T"
res = [get_mut_type(item) for item in mut_string.split(',')]
res

['C>T', 'C>T', 'G>A', 'C>T', 'C>T', 'C>T', 'G>A', 'C>T', 'C>T']

In [ ]:
df["Counts"] =df["subs"].apply(lambda x: dict(Counter(x.split(","))))
df["Counts"]


In [9]:
def get_context(genome_seq, mutation):
    if isinstance(mutation, str) and len(mutation) >= 2:
        pos = int(mutation[1:-1]) - 1  # -1 as Python uses 0-based indexing
        context = genome_seq[pos-1:pos+2]  # get the base before and after
        return context
    else:
        return ''
df["context"]=df["privateNucMutations.unlabeledSubstitutions"].apply(lambda x: ','.join([get_context(reference_genome, item) for item in x.split(',')])if isinstance(x, str) else '')
df["context"]   

0                                                     CAC,ATA
1           ACT,ACA,AGT,ACA,TCT,GAT,CTT,CGT,AGT,CGC,AAA,AC...
2                                                            
3                                                         ATG
4                         CCT,CGA,TCT,CAG,TCA,TCA,ACC,CGA,ACG
                                  ...                        
16921560                                              CCA,ACT
16921561                                                  ACG
16921562                                          ACA,ACA,GCT
16921563                                                  ACT
16921564                                          TGT,AGG,ACT
Name: context, Length: 16921565, dtype: object

In [10]:
df[df['context'].apply(lambda x: isinstance(x, str) and len(x) < 2)].head()

,seqName,privateNucMutations.unlabeledSubstitutions,subs,Counts,context
2,hCoV-19/South Korea/KDCA36211s/2022|2022-11-16...,NaN,,{'': 1},
11,hCoV-19/England/LSPA-3699AD2/2022|2022-02-10|2...,NaN,,{'': 1},
19,hCoV-19/USA/WA-UW-2317/2020|2020-03-30|2020-06-29,NaN,,{'': 1},
20,hCoV-19/USA/AZ-TG1278416/2021|2021-12-03|2022-...,NaN,,{'': 1},
24,hCoV-19/USA/CT-CDC-QDX26076006/2021|2021-06-19...,NaN,,{'': 1},


In [11]:
def spectrum(subs, contexts):
    if not isinstance(subs, str) or not isinstance(contexts, str):
        return ''
    if not subs.strip() or not contexts.strip():
        return ''
    subs= subs.split(',')
    contexts = contexts.split(',')
    spectra = []
    for mutation, context in zip(subs, contexts): 
        if len(context)<2:
            continue
        spectra.append(f"{context[0]}[{mutation}]{context[-1]}")
    return ','.join(spectra)
   
df["spectrum"] = df.apply(lambda row: spectrum(row["subs"], row["context"]), axis=1)
df["spectrum"].head()



0                                      C[A>G]C,A[T>C]A
1    A[C>T]T,A[C>T]A,A[G>T]T,A[C>T]A,T[C>T]T,G[A>G]...
2                                                     
3                                              A[T>C]G
4    C[C>A]T,C[G>T]A,T[C>A]T,C[A>G]G,T[C>T]A,T[C>T]...
Name: spectrum, dtype: object

In [12]:
#Get Counts per substitution context
def count_GtoA(spectrum):
    counts = Counter()
    if not isinstance(spectrum, str):
        return {}
    muts = spectrum.split(",")
    for mut in muts:
        if mut[2:5] == 'G>A':
            counts[mut]+=1
    return counts

def count_AtoG(spectrum):
    counts = Counter()
    if not isinstance(spectrum, str):
        return {}
    muts = spectrum.split(",")
    for mut in muts:
        if mut[2:5] == 'A>G':
            counts[mut]+=1
    return counts

def count_CtoT(spectrum):
    counts = Counter()
    if not isinstance(spectrum, str):
        return {}
    muts = spectrum.split(",")
    for mut in muts:
        if mut[2:5] == 'C>T':
            counts[mut]+=1
    return counts

def count_TtoC(spectrum):
    counts = Counter()
    if not isinstance(spectrum, str):
        return {}
    muts = spectrum.split(",")
    for mut in muts:
        if mut[2:5] == 'T>C':
            counts[mut]+=1
    return counts

df["G>A_counts"] = df["spectrum"].apply(count_GtoA)
df["A>G_counts"] = df["spectrum"].apply(count_AtoG)
df["C>T_counts"] = df["spectrum"].apply(count_CtoT)
df["T>C_counts"] = df["spectrum"].apply(count_TtoC)
print(df["G>A_counts"].head())
print(df["A>G_counts"].head())
print(df["C>T_counts"].head())
print(df["T>C_counts"].head())

 

0                {}
1    {'A[G>A]T': 1}
2                {}
3                {}
4                {}
Name: G>A_counts, dtype: object
0    {'C[A>G]C': 1}
1    {'G[A>G]T': 1}
2                {}
3                {}
4    {'C[A>G]G': 1}
Name: A>G_counts, dtype: object
0                                                   {}
1    {'A[C>T]T': 1, 'A[C>T]A': 2, 'T[C>T]T': 1, 'A[...
2                                                   {}
3                                                   {}
4           {'T[C>T]A': 2, 'A[C>T]C': 1, 'A[C>T]G': 1}
Name: C>T_counts, dtype: object
0    {'A[T>C]A': 1}
1    {'A[T>C]A': 1}
2                {}
3    {'A[T>C]G': 1}
4                {}
Name: T>C_counts, dtype: object


In [13]:
# Get Proportions per substitution context
def get_proportion(df):
    dict = {}
    total = sum(df.values())
    for key, value in df.items():
        dict[key] = value/total
       
    return dict

df["G>Aproportions"] = df.apply(lambda row: get_proportion(row["G>A_counts"]), axis=1)
df["A>Gproportions"] = df.apply(lambda row: get_proportion(row["A>G_counts"]), axis=1)
df["C>Tproportions"] = df.apply(lambda row: get_proportion(row["C>T_counts"]), axis=1)
df["T>Cproportions"] = df.apply(lambda row: get_proportion(row["T>C_counts"]), axis=1)

print(df["G>Aproportions"].head())
print(df["A>Gproportions"].head())
print(df["C>Tproportions"].head())
print(df["T>Cproportions"].head())


0                  {}
1    {'A[G>A]T': 1.0}
2                  {}
3                  {}
4                  {}
Name: G>Aproportions, dtype: object
0    {'C[A>G]C': 1.0}
1    {'G[A>G]T': 1.0}
2                  {}
3                  {}
4    {'C[A>G]G': 1.0}
Name: A>Gproportions, dtype: object
0                                                   {}
1    {'A[C>T]T': 0.16666666666666666, 'A[C>T]A': 0....
2                                                   {}
3                                                   {}
4    {'T[C>T]A': 0.5, 'A[C>T]C': 0.25, 'A[C>T]G': 0...
Name: C>Tproportions, dtype: object
0    {'A[T>C]A': 1.0}
1    {'A[T>C]A': 1.0}
2                  {}
3    {'A[T>C]G': 1.0}
4                  {}
Name: T>Cproportions, dtype: object


In [14]:
# Because df["Counts"] is not read as a dict in the next step

df["Counts"] = (
    df["Counts"]
    .astype(str)
    .str.replace("Counter(", "")  # remove "Counter("
    .str.rstrip(")")                           # remove trailing ")"
    .apply(lambda x: ast.literal_eval(x) if x not in ["", "nan", "Counter"] else {})
)
print(df["Counts"].head(20))
print(type(df["Counts"]))

0                                  {'A>G': 1, 'T>C': 1}
1     {'C>T': 6, 'G>T': 4, 'A>G': 1, 'T>G': 1, 'G>A'...
2                                               {'': 1}
3                                            {'T>C': 1}
4              {'C>A': 2, 'G>T': 2, 'A>G': 1, 'C>T': 4}
5                        {'A>G': 1, 'T>C': 1, 'T>A': 1}
6                                            {'C>T': 2}
7     {'C>T': 4, 'T>A': 1, 'G>T': 2, 'G>A': 1, 'C>G'...
8     {'A>G': 2, 'C>T': 5, 'G>T': 2, 'G>A': 1, 'T>C'...
9                                            {'T>C': 1}
10                                           {'C>T': 1}
11                                              {'': 1}
12             {'A>G': 1, 'A>C': 1, 'G>T': 1, 'T>C': 1}
13                                 {'A>G': 1, 'T>G': 1}
14                                 {'C>T': 3, 'T>C': 1}
15    {'G>A': 2, 'C>T': 1, 'A>G': 1, 'T>G': 1, 'A>C'...
16                       {'A>G': 1, 'G>T': 1, 'A>T': 1}
17                                 {'C>T': 3, 'G

In [17]:
def get_likelihood_ratio(counts,pM,pN):
    counts=np.array(counts,dtype=float)
    llM= float(multinomial.logpmf(counts, n=np.sum(counts), p=pM))
    llN = float(multinomial.logpmf(counts, n=np.sum(counts), p=pN))
    llr=llM-llN
    return llr
probs_df = pd.read_csv("/Users/reem/Downloads/estimated_mutation_distribution.tsv", delimiter="\t")
#probs_df

pM=probs_df["Molnupiravir"].values.tolist()
pN=probs_df["Normal"].values.tolist()
mutation_types = probs_df["MutationType"].str.replace("→", ">").tolist()


llr_list=[]
for i,row in df.iterrows():
    counts_dict = row["Counts"]
    counts = [counts_dict.get(mut,0) for mut in mutation_types]
    llr = get_likelihood_ratio(counts,pM,pN)
    llr_list.append(llr)
df["LLR"] = llr_list
df["LLR"].head()


0   -0.739348
1   -6.493157
2    0.000000
3   -0.528901
4   -4.058362
Name: LLR, dtype: float64

In [18]:
print(df["LLR"].min())
print(df["LLR"].max())

-6208.080529299026
32.21142423519336


In [19]:
df.to_csv("/Users/reem/Mov/final_results_LLRs.tsv",sep="\t")